# Soluciones — Clase 24: Series de tiempo — datos temporales

> **Nota para el docente:** Soluciones completas para todos los ejercicios y el homework. No compartir con alumnos antes de la entrega.

In [ ]:
# Importaciones completas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 4)
print('Librerías cargadas ✅')

## Solución — Generar y preparar datos

In [ ]:
# Generar datos simulados de ventas diarias con tendencia y estacionalidad
# Los datos tienen tendencia creciente y pico en meses de verano
np.random.seed(42)
fechas = pd.date_range('2022-01-01', '2023-12-31', freq='D')
n = len(fechas)

tendencia    = np.linspace(1000, 1500, n)
estacional   = 250 * np.sin(2 * np.pi * np.arange(n) / 365 - np.pi/2)
ruido        = np.random.normal(0, 90, n)
fin_de_semana = np.array([1.25 if f.dayofweek >= 5 else 1.0 for f in fechas])

ventas = (tendencia + estacional + ruido) * fin_de_semana
ventas = np.clip(ventas, 200, None).round(0)

df = pd.DataFrame({'ventas': ventas}, index=fechas)
df.index.name = 'fecha'
col_ventas = 'ventas'

print(f'Filas: {len(df)}')
print(f'Rango: {df.index.min().date()} → {df.index.max().date()}')
print(f'Ventas promedio diario: {df[col_ventas].mean():,.0f}')
df.describe().round(0)

## Solución Ejercicio 3 — Resample correcto

In [ ]:
# Resample por diferentes periodos
# CORRECTO: usar resample, no groupby con dt.month
ventas_mensuales = df[col_ventas].resample('M').sum()
ventas_semanales = df[col_ventas].resample('W').sum()

print('Ventas mensuales:')
for fecha, valor in ventas_mensuales.items():
    barra = '█' * int(valor / 10000)
    print(f'  {fecha.strftime("%b %Y")}: {valor:>8,.0f}')

print(f'\nMes más alto:  {ventas_mensuales.idxmax().strftime("%B %Y")}')
print(f'Mes más bajo:  {ventas_mensuales.idxmin().strftime("%B %Y")}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ventas_mensuales.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='k')
axes[0].set_title('Ventas por mes')
axes[0].set_xlabel('Mes')
axes[0].tick_params(axis='x', rotation=45)
axes[0].axhline(ventas_mensuales.mean(), color='red', linestyle='--', label='Promedio')
axes[0].legend()

ventas_semanales.plot(ax=axes[1], color='green', alpha=0.7, linewidth=1.5)
axes[1].set_title('Ventas semanales')
axes[1].set_xlabel('Semana')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Solución Ejercicio 4 — Promedio móvil con visualización completa

In [ ]:
# Calcular promedios móviles y comparar
df['media_7d']  = df[col_ventas].rolling(7).mean()
df['media_30d'] = df[col_ventas].rolling(30).mean()

print(f'Primeros NaN en media_7d: {df["media_7d"].isna().sum()} (los primeros 6 días)')
print(f'Primeros NaN en media_30d: {df["media_30d"].isna().sum()} (los primeros 29 días)')

fig, ax = plt.subplots(figsize=(14, 5))

ax.fill_between(df.index, df[col_ventas], alpha=0.1, color='steelblue')
ax.plot(df.index, df[col_ventas],
        label='Ventas diarias', color='steelblue', alpha=0.4, linewidth=0.8)
ax.plot(df.index, df['media_7d'],
        label='Media 7 días', color='red', linewidth=1.8, alpha=0.9)
ax.plot(df.index, df['media_30d'],
        label='Media 30 días', color='darkgreen', linewidth=2.5)

ax.set_xlabel('Fecha')
ax.set_ylabel('Ventas ($)')
ax.set_title('Ventas diarias con promedios móviles — 2022-2023')
ax.legend()
ax.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('\n► La línea verde (30 días) muestra claramente la tendencia creciente')
print('► Los picos del verano son visibles incluso en la media de 30 días')
print('► La media de 7 días sigue más el ruido pero suaviza los días individuales')

## Solución Ejercicio 6 — Descomposición estacional completa

In [ ]:
# Descomposición estacional
from statsmodels.tsa.seasonal import seasonal_decompose

serie = ventas_mensuales.dropna()
resultado = seasonal_decompose(serie, model='additive', period=12)

componentes = {
    'Serie original':  (resultado.observed, 'steelblue'),
    'Tendencia':       (resultado.trend,    'darkgreen'),
    'Estacionalidad':  (resultado.seasonal, 'orange'),
    'Residuo':         (resultado.resid,    'gray')
}

fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True)
for ax, (titulo, (datos, color)) in zip(axes, componentes.items()):
    datos.plot(ax=ax, color=color, linewidth=1.5)
    ax.set_title(titulo, fontsize=11, pad=3)
    ax.grid(alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Descomposición estacional — Ventas mensuales (2022-2023)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

# Analizar los componentes
est = resultado.seasonal.dropna()
mes_pico = est.groupby(est.index.month).mean()
print('\nContribución estacional promedio por mes:')
meses_nombre = ['Ene','Feb','Mar','Abr','May','Jun',
                'Jul','Ago','Sep','Oct','Nov','Dic']
for i, (m, v) in enumerate(mes_pico.items()):
    signo = '+' if v > 0 else ''
    print(f'  {meses_nombre[i]:3s}: {signo}{v:>8,.0f}')

## Solución Ejercicio 7 — Pronóstico comparativo con evaluación

In [ ]:
# Pronósticos y evaluación usando los últimos 3 meses como validación
# Separamos el último cuarto para testear los pronósticos
entrenamiento = serie.iloc[:-3]
real          = serie.iloc[-3:]

# Calcular pronósticos a partir del set de entrenamiento
p_naive    = entrenamiento.iloc[-1]
p_ma3      = entrenamiento.tail(3).mean()
p_seasonal = entrenamiento.iloc[-12] if len(entrenamiento) >= 12 else np.nan

valor_real_siguiente = real.iloc[0]

print(f'Valor real (siguiente mes): {valor_real_siguiente:,.0f}')
print(f'Naive:            {p_naive:>10,.0f}  |  Error: {abs(p_naive - valor_real_siguiente):>8,.0f}')
print(f'Media móvil 3m:   {p_ma3:>10,.0f}  |  Error: {abs(p_ma3 - valor_real_siguiente):>8,.0f}')
if not np.isnan(p_seasonal):
    print(f'Seasonal naive:   {p_seasonal:>10,.0f}  |  Error: {abs(p_seasonal - valor_real_siguiente):>8,.0f}')

# Visualización
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(entrenamiento.index, entrenamiento.values,
        marker='o', color='steelblue', label='Histórico', linewidth=2)
ax.plot(real.index, real.values,
        marker='s', color='black', linewidth=2, label='Valores reales')

prox = real.index[0]
for nombre, valor, color in [
    ('Naive', p_naive, '#E53935'),
    ('MA-3', p_ma3, '#43A047')
]:
    ax.scatter(prox, valor, s=180, color=color, zorder=6, label=f'{nombre}: {valor:,.0f}')

ax.set_title('Pronósticos vs valores reales')
ax.set_xlabel('Mes')
ax.set_ylabel('Ventas')
ax.legend()
ax.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('\n► Con tendencia creciente, el MA-3 suele ser mejor que el naive')
print('► Con estacionalidad fuerte, el seasonal naive puede ganar en los meses de pico')